# 00 · Environment Setup & Data Ingestion

**Goal:** Establish a reproducible Google Colab workspace, configure secure connections (SSH/GitHub), initialize the DVC pipeline, and retrieve the raw dataset.

**Output:** A fully configured MLOps environment with a populated `data/raw/` directory, ready for downstream profiling, feature engineering, and modeling.

---

### 📑 Table of Contents

| Step | Section | Description |
|:---:|---|---|
| **0** | 🚀 **Bootstrap** | Mount Google Drive, configure SSH keys, and clone/update the repository. |
| **1** | 📦 **Environment Init** | Install dependencies from `pyproject.toml` and configure the local DVC remote. |
| **2** | 📥 **DVC Pull** | Retrieve tracked datasets and artifacts from the Google Drive remote. |
| **3** | 🔄 **Fallback Ingestion** | Download raw data via Kaggle API and push to DVC (only if missing). |
| **4** | 🔍 **Verification** | Verify paths, check DVC pipeline status, and prepare for execution. |
| **5** | ⚙️ **Pipeline Execution** | Run the full MLOps pipeline (`dvc repro`). |

---

## 0. 🚀 Bootstrap: Drive, SSH & Clone

In [2]:
# ===================================================================
# 1. BOOTSTRAP: Mount Drive, Setup SSH, and Clone Repo
# ===================================================================
import sys
import os
import subprocess

# A. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# B. Setup SSH Key from Drive
ssh_dir = os.path.expanduser('~/.ssh')
os.makedirs(ssh_dir, exist_ok=True)
key_source = '/content/drive/MyDrive/ssh_config/housing_key'
key_dest = os.path.join(ssh_dir, 'id_rsa')

use_ssh = False
if os.path.exists(key_source):
    try:
        subprocess.run(f'cp {key_source} {key_dest}', shell=True, check=True)
        subprocess.run(f'chmod 600 {key_dest}', shell=True, check=True)
        
        subprocess.run('ssh-keyscan -H github.com >> ~/.ssh/known_hosts', shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        print("✅ SSH Key configured.")
        use_ssh = True
    except subprocess.CalledProcessError:
        print("⚠️ SSH Key found but failed to configure. Falling back to HTTPS.")
        use_ssh = False
else:
    print(f"⚠️ SSH Key not found at {key_source}. Falling back to HTTPS.")

# C. Clone / Update Repository
repo_owner = "ebramrafat653-wq"
repo_name = "california_housing_full_project"
repo_dir = f"/content/{repo_name}"

if use_ssh:
    repo_url = f"git@github.com:{repo_owner}/{repo_name}.git"
else:
    repo_url = f"https://github.com/{repo_owner}/{repo_name}.git"

try:
    if not os.path.exists(repo_dir):
        print(f"📥 Cloning repository from {repo_url}...")
        subprocess.run(f'git clone {repo_url} {repo_dir}', shell=True, check=True)
    else:
        print("✅ Repository exists. Pulling latest changes...")
        subprocess.run(f'git -C {repo_dir} pull --rebase', shell=True, check=True)
    
    # D. Change directory and add to Python Path
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)  
    print(f"📍 Working directory set to: {os.getcwd()}")

except subprocess.CalledProcessError as e:
    print(f"❌ Git operation failed: {e}")
    print("Please check your internet connection or verify the repository URL.")
    raise

Mounted at /content/drive
✅ SSH Key configured.
📥 Cloning repository from git@github.com:ebramrafat653-wq/california_housing_full_project.git...
📍 Working directory set to: /content/california_housing_full_project


## 1. 📦 Install Dependencies & Configure DVC Remote

In [3]:
# ===================================================================
# 2. INSTALL DEPENDENCIES & CONFIGURE DVC LOCAL REMOTE
# ===================================================================
# Install project + dev tools (ruff, pytest, etc.)

%pip install -e ".[dev]"  

# Import and configure local DVC remote (writes to .dvc/config.local)
from src.utils.colab_setup import configure_dvc_local
from src.utils.paths import PROJECT_DIR

configure_dvc_local(PROJECT_DIR)
print("✅ Environment configured successfully.")

Obtaining file:///content/california_housing_full_project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 74.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 84.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.2/257.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 29.

/content/california_housing_full_project/src/utils/paths.py:28: RuntimeWarning: Logger auto-initialized with defaults. Call setup_logging() explicitly at startup for full control.
  logger = get_logger(__name__)


2026-09-02 20:54:17 | INFO     | src.utils.colab_setup | ✅ DVC local remote configured in .dvc/config.local
2026-09-02 20:54:17 | INFO     | src.utils.colab_setup |    (This file is ignored by Git)
✅ Environment configured successfully.


## 2. 📥 Pull Data from DVC Remote (Resilient)

In [4]:
# ===================================================================
# 3. PULL DATA FROM DVC REMOTE (RESILIENT)
# ===================================================================
import subprocess
import os
import sys

print("📦 Pulling data from DVC remote (Google Drive)...")

def dvc_pull_with_fallback():
    """
    Attempt to pull data with automatic fallback to --force
    if the lockfile mismatch causes failure.
    """
    try:
        # First attempt: normal pull
        subprocess.run(["dvc", "pull"], check=True, text=True)
        return True, "✅ Data pulled successfully (clean)."
    except subprocess.CalledProcessError as e:
        print(f"⚠️ DVC pull failed with error code {e.returncode}. Trying --force...")
        try:
            # Second attempt: pull with --force (forces sync even if lockfile differs)
            subprocess.run(["dvc", "pull", "--force"], check=True, text=True)
            return True, "✅ Data pulled successfully (with --force)."
        except subprocess.CalledProcessError as e2:
            # Even --force failed
            return False, f"❌ DVC pull failed even with --force (error {e2.returncode})."
    except FileNotFoundError:
        return False, "❌ DVC is not installed. Please check installation."

# Execute the function
success, message = dvc_pull_with_fallback()
print(message)

# ===================================================================
# 3b. Verify essential data files exist (even if pull was incomplete)
# ===================================================================
essential_files = [
    "data/raw/housing.csv",
    "data/processed/train_clean.csv",
    "data/processed/test_clean.csv"
]

missing = [f for f in essential_files if not os.path.exists(f)]

if missing:
    print("⚠️ Some essential files are missing after the pull attempt:")
    for f in missing:
        print(f"   - {f}")
    print("🔧 You may need to run `dvc repro` to generate them locally (if data exists in the remote).")
else:
    print("✅ All essential files are present.")

# ===================================================================
# 3c. Inform about empty directories (won't block execution)
# ===================================================================
empty_dirs = ["models", "reports/validation"]
for d in empty_dirs:
    if not os.path.exists(d) or not os.listdir(d):
        print(f"ℹ️  Directory '{d}' is missing or empty (not an issue if you haven't used it yet).")

print("\n🚀 You can safely proceed to Cell 4 (dvc repro).")

📦 Pulling data from DVC remote (Google Drive)...
⚠️ DVC pull failed with error code 1. Trying --force...
❌ DVC pull failed even with --force (error 1).
✅ All essential files are present.
ℹ️  Directory 'models' is missing or empty (not an issue if you haven't used it yet).
ℹ️  Directory 'reports/validation' is missing or empty (not an issue if you haven't used it yet).

🚀 You can safely proceed to Cell 4 (dvc repro).


## 3. 🌾 Initial Data Ingestion (Kaggle Fallback)

In [5]:
# ===================================================================
# 4. INITIAL DATA INGESTION (Run ONLY if housing.csv is missing)
# ===================================================================
import os
import subprocess
from pathlib import Path

raw_csv = Path("data/raw/housing.csv")

if not raw_csv.exists():
    print("⚠️ housing.csv not found in DVC remote or locally. Running initial ingestion from Kaggle...")
    
    try:
        # 1. Run the ingestion script
        subprocess.run(
            ["python", "-m", "src.data.ingestion", "--no-dvc"],
            check=True,
            text=True
        )
        print("✅ Ingestion script completed successfully.")
        
        # 2. Track the generated file with DVC
        print("📦 Tracking data with DVC...")
        subprocess.run(["dvc", "add", "data/raw/housing.csv"], check=True, text=True)
        
        # 3. Push to Google Drive remote
        print("⬆️ Pushing to DVC remote (Google Drive)...")
        subprocess.run(["dvc", "push"], check=True, text=True)
        
        print("✅ Data ingested, tracked, and pushed to Google Drive successfully.")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ An error occurred during bootstrap (exit code: {e.returncode}).")
        print("Please check the logs above. You may need to run `dvc repro` manually later.")
        # Don't raise here, let the notebook continue so the user can debug.
        
else:
    size_mb = raw_csv.stat().st_size / (1024 * 1024)
    print(f"✅ housing.csv already present ({size_mb:.2f} MB). Skipping ingestion.")
    print("ℹ️  DVC will verify this file in the next cell (dvc repro).")

✅ housing.csv already present (1.36 MB). Skipping ingestion.
ℹ️  DVC will verify this file in the next cell (dvc repro).


## 4. 🔍 Environment Verification & Summary

In [6]:
# ===================================================================
# 5. VERIFY ENVIRONMENT & NEXT STEPS
# ===================================================================
import subprocess
from src.utils.paths import verify_paths

# 1. Print diagnostic table of all paths
verify_paths()

# 2. Check DVC pipeline status
print("\n🔍 DVC Pipeline Status:")
status_result = subprocess.run(["dvc", "status"], capture_output=True, text=True)
print(status_result.stdout)

if status_result.stderr:
    print(f"⚠️ DVC status warnings:\n{status_result.stderr}")

# 3. Smart recommendation based on status
print("\n" + "="*60)
if "is missing" in status_result.stdout or "changed" in status_result.stdout:
    print("  ⚠️  CHANGES DETECTED in pipeline outputs.")
    print("  ▶  Run `dvc repro` to regenerate the missing/changed files.")
elif "up-to-date" in status_result.stdout:
    print("  ✅  PIPELINE IS UP-TO-DATE (No changes needed).")
    print("  ▶  You can skip `dvc repro` or run it to force execution.")
else:
    print("  ✅  ENVIRONMENT READY")
    print("  ▶  To run the full ML pipeline, execute:")
    print("  >>> !dvc repro")

print("="*60)

  Environment  : Google Colab
  Project root : /content/california_housing_full_project
  Drive base   : /content/drive/MyDrive
  Data root    : /content/california_housing_full_project/data
  DVC remote   : mylocal  →  /content/drive/MyDrive/dvc_storage
  DVC ready    : True
  ✓  raw            /content/california_housing_full_project/data/raw
  ✓  interim        /content/california_housing_full_project/data/interim
  ✓  processed      /content/california_housing_full_project/data/processed
  ✗  models         /content/california_housing_full_project/models
  ✓  artifacts      /content/california_housing_full_project/artifacts
  ✓  kaggle_json    /content/drive/MyDrive/kaggle.json
  ✓  configs        /content/california_housing_full_project/configs
  ✓  notebooks      /content/california_housing_full_project/notebooks
  ✓  reports        /content/california_housing_full_project/reports
  ✓  src            /content/california_housing_full_project/src

🔍 DVC Pipeline Status:
validation:

## 5. ▶️ Run the Full ML Pipeline

In [7]:
# ===================================================================
# 6. RUN THE FULL ML PIPELINE
# ===================================================================
!dvc repro preprocessing

Stage 'ingestion' didn't change, skipping
Stage 'splitting' didn't change, skipping
Stage 'cleaning' didn't change, skipping
Stage 'engineering' didn't change, skipping
Stage 'preprocessing' didn't change, skipping
Data and pipelines are up to date.
